In [1]:
import numpy as np
import pandas as pd
import glob
import os
import random
import matplotlib.pyplot as plt

In [2]:
## grab files
directory = 'optseq2_design_ok'
# pattern = 'tr-200_min-3_nsearch-1000*.par'
pattern = 'design_12cond_focb_ok-*.par'
texture_id = [i for i in range(160)] # there are 160 density-4 snakes
target_directory = 'trial_sequences'
# define mapping frames to ms
# frames_ms_mapping = {0 : 0, 2 : 17, 4 : 33, 8 : 67, 16 : 134, 32 : 267, 64 : 533}
ms = [0,25, 50, 75, 100, 125, 175, 250, 350, 525, 750] # Pilot 1: [0, 25,50,75,100,175,275,475,750,1225,2000]
frames = [0, 3, 6, 9, 12, 15, 21, 30, 42, 63, 90] # Pilot 1: [0,3,6,9,12,21,33,57,90,147,240]
frames_ms_mapping = {frame: ms for frame, ms in zip(frames, ms)}

dur_idx_frames_mapping = {idx : frame for idx, frame in zip(range(12), [0] + frames)}

files = glob.glob(os.path.join(directory, pattern))
files.sort()

plot = False
if plot:
    fig, axs = plt.subplots(3,5, figsize=(15,9))

# print(files)
for j, file in enumerate(files):

    with open(file, 'r') as f:
        lines = f.readlines()

    rows = []
    for line in lines:
        split_line = line.split(' ')
        vals = [val.strip() for val in split_line if (val != '') & (val != '\n')]
        for i in range(len(vals)-1):
            vals[i] = np.round(float(vals[i]), 2)
        
        # adding 0s to itis
        if vals[-1] == 'NULL':
            rows[-1][2] = np.round(rows[-1][2] + vals[2], 2)
            # uncomment to debug
    #         rows.append(vals)
        
        else:
            rows.append(vals)
    print(rows)
    iti_df = pd.DataFrame(rows, columns = ['time', 'id', 'iti_s', 'unknown', 'type'])
    iti_df['time'] = iti_df['time'].astype(float)
    iti_df['iti_s'] = iti_df['iti_s'].astype(float)
    iti_df['iti_TR'] = np.round(iti_df['iti_s']/.9).astype(int)
    iti_df['iti_TR_cum'] = np.cumsum(iti_df['iti_TR'])
    iti_df['cond_frames'] = [dur_idx_frames_mapping[int(val.split('_')[1])] for val in iti_df['type']]
    iti_df['type'] = [val.split('_')[0] for val in iti_df['type']]
    iti_df = iti_df.drop(['id', 'unknown'], axis = 1)
    iti_df['cond_ms'] = [frames_ms_mapping[val] for val in iti_df['cond_frames']]
    iti_df['phase_0_ms'] = [2000] * len(iti_df)
    iti_df['phase_0_frames'] = [240] * len(iti_df)
    iti_df['phase_1_ms'] = iti_df['iti_s'] - 2.0
    iti_df['phase_1_frames'] = (iti_df['phase_1_ms'] * 120).astype(int)
      
    random.shuffle(texture_id)
    iti_df['texture_id'] = texture_id[:len(iti_df)]

    # check
    print(iti_df.iti_TR_cum.iloc[-1])
    if plot:

        n, bins, _ = axs.flatten()[j].hist(iti_df['iti_s'])
        if j % 5 ==0:
            axs.flatten()[j].set_ylabel("counts")
        if j >= 10:
            axs.flatten()[j].set_xlabel("ITI [s]")
        
    # dot product of histogram
    # print(iti_df.iti_TR.value_counts().values @ iti_df.iti_TR.value_counts().keys().to_numpy())    
    # print(n.shape, bins.shape)
    # print(n, bins)
    
    
    # save
    try:
        os.mkdir(target_directory)
    except FileExistsError:
        pass
    path = os.path.join(target_directory, f'design_12cond_focb_ok_{file.split("-")[-1].split(".")[0]}.csv')
    iti_df.to_csv(path, index = False)
# plt.savefig('iti_hist_new_seq.png', transparent = False)

[[0.0, 3.0, 3.6, 1.0, 'dur_2'], [3.6, 2.0, 3.6, 1.0, 'dur_1'], [7.2, 10.0, 3.6, 1.0, 'dur_9'], [10.8, 3.0, 3.6, 1.0, 'dur_2'], [14.4, 5.0, 3.6, 1.0, 'dur_4'], [18.0, 8.0, 3.6, 1.0, 'dur_7'], [21.6, 6.0, 3.6, 1.0, 'dur_5'], [25.2, 1.0, 3.6, 1.0, 'dur_0'], [28.8, 8.0, 3.6, 1.0, 'dur_7'], [32.4, 9.0, 3.6, 1.0, 'dur_8'], [36.0, 9.0, 3.6, 1.0, 'dur_8'], [39.6, 3.0, 3.6, 1.0, 'dur_2'], [43.2, 7.0, 3.6, 1.0, 'dur_6'], [46.8, 6.0, 5.4, 1.0, 'dur_5'], [52.2, 12.0, 3.6, 1.0, 'dur_11'], [55.8, 6.0, 3.6, 1.0, 'dur_5'], [59.4, 4.0, 3.6, 1.0, 'dur_3'], [63.0, 8.0, 3.6, 1.0, 'dur_7'], [66.6, 2.0, 3.6, 1.0, 'dur_1'], [70.2, 4.0, 3.6, 1.0, 'dur_3'], [73.8, 2.0, 3.6, 1.0, 'dur_1'], [77.4, 8.0, 6.3, 1.0, 'dur_7'], [83.7, 3.0, 7.2, 1.0, 'dur_2'], [90.9, 7.0, 3.6, 1.0, 'dur_6'], [94.5, 5.0, 3.6, 1.0, 'dur_4'], [98.1, 9.0, 4.5, 1.0, 'dur_8'], [102.6, 5.0, 3.6, 1.0, 'dur_4'], [106.2, 4.0, 3.6, 1.0, 'dur_3'], [109.8, 4.0, 4.5, 1.0, 'dur_3'], [114.3, 12.0, 3.6, 1.0, 'dur_11'], [117.9, 7.0, 3.6, 1.0, 'dur_6'], 